# Microsoft-Decision-1 Hands-on

## サマリー

テキストや JSON から、文章ではなく確率・選択肢・スコアを取得します。

| Part | 内容 |
| --- | --- |
| 0 | インストール、接続設定、認証、最小限の呼び出し関数、グラフの準備 |
| 1 | `noul`、判定基準、`choice`、`score`、スコア計算、生の JSON |
| 2 | JSON 入力、会話配列、複数質問、説明が `null` の選択肢 |

- Foundry リソースと `Microsoft-Decision-1` のデプロイが必要です。`model` にはデプロイ名を指定します。
- 上から順に実行してください。後のセルは、前のセルで定義した質問や設定を使用します。
- 判定結果は JSON と確率分布のグラフで確認できます。描画には取得済みの結果を使い、追加の API 呼び出しは行いません。
- 全セルが成功した場合、モデルへのリクエストは計 9 回です。各呼び出しは課金対象です。2-2 と 2-4 は Foundry の公式文書だけでは詳細を確認できない形式のため、解説の注意事項を確認してください。
- 判定値はモデルの推定です。日本語での精度や特定の数値を保証するものではありません。

出典: [Microsoft-Decision-1](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision)

## 全体像: 作家ではなく審査員

GPT のような生成モデルは、文章を書く作家です。Decision-1 は、渡された材料を見て札を上げる審査員です。返ってくるのは文章ではなく、アプリケーションがそのまま分岐に使える値です。公式ドキュメントは、同じ入力に対する、範囲の決まった型のある判断が必要なときに使うよう案内しています。

| 質問タイプ | たとえ | 質問の書き方 | 返ってくる値 |
| --- | --- | --- | --- |
| `noul` | 降水確率。明日は雨が降るか | `instructions`、任意で `criteria` の `true` と `false` | `noul`: yes である確率（0 から 1） |
| `choice` | 郵便物の仕分け箱。どの箱に入れるか | `criteria`（選択肢 → 説明） | `choice`、各選択肢の `probabilities`、`confidence` |
| `score` | 辛さの目盛り。どの段階か | `criteria`（低い順に並べたラベルの配列） | `score`（段階の添字の確率加重平均）、`legend`、各段階の `probabilities`、`confidence` |

出典: 質問タイプとリクエストの形は https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#choose-a-question-type 。`probabilities`、`legend`、`confidence` の具体的なフィールドは https://learn.microsoft.com/azure/databricks/machine-learning/model-serving/query-systemone-models#response-formatと LiveKit の実装による。

```mermaid
flowchart LR
    APP[アプリケーション] -->|"model, state, questions"| EP["POST エンドポイント/providers/microsoft/v1/systemone"]
    EP --> R[レスポンス]
    R --> M[model]
    R --> A["answers: 質問名 → 回答"]
    R --> U["usage: input_tokens, output_tokens"]
    A --> N["noul: 確率"]
    A --> C["choice: 選択肢と各確率"]
    A --> S["score: 加重平均と各段階の確率"]
```

### 生成モデルとの組み合わせ

公式ドキュメントは、Decision-1 で振り分けや関門の判定をしてから、承認されたリクエストだけを生成モデルに渡す使い方を挙げています。高頻度の入口処理を軽い判定で済ませ、文章が必要な場面だけ生成モデルを使うイメージです。

```mermaid
flowchart LR
    IN[入力] --> D["Decision-1: 関門 / 振り分け"]
    D -->|通過| G[生成モデル: 回答の作成]
    D -->|"確率が低い / 判断不能"| H[人間のレビュー]
    D -->|対象外| X[自動応答で終了]
```

出典: https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#when-to-use-microsoft-decision-1

### Azure OpenAI の Decisions API（`gpt-6-luna`）との違い

| 項目 | Decisions API（`gpt-6-luna`） | Microsoft-Decision-1 |
| --- | --- | --- |
| エンドポイント | `https://{リソース}.openai.azure.com/openai/v1/decisions` | `{Foundry リソースのエンドポイント}/providers/microsoft/v1/systemone` |
| 入力 | `input`（文字列、または `input_text` と `input_image` を含む user メッセージ） | `state`（テキストまたは JSON） |
| `questions` | 配列 | オブジェクト（質問名がキー） |
| 真偽の質問 | `predicate` | `noul` |
| 選択肢 | `choices: [{value, description}]` | `criteria: {選択肢: 説明}` |
| 段階 | `levels: [{label, description}]` | `criteria: [ラベル, ...]` |
| 回答 | `answers` は配列で、質問と同じ順序 | `answers` はオブジェクトで、質問名がキー |

出典: Decisions API は https://learn.microsoft.com/azure/foundry/openai/how-to/decisions 、Decision-1 は上記の Learn ページ。

## Part 0: セットアップ

### 0-1. 必要なライブラリ

`httpx` は HTTP 呼び出し、`azure-identity` は Microsoft Entra ID 認証、`python-dotenv` は既存の `.env` の読み込み、`matplotlib` はグラフの描画に使用します。独自の型定義は不要です。必要であればインストール後にカーネルを再起動してください。

出典: [前提条件](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#prerequisites)

In [ ]:
%pip install -q httpx azure-identity python-dotenv matplotlib

### 0-2. 接続設定

両ノートブックで同じ `.env` を使用します。[.env.example](./.env.example) を同じフォルダーに `.env` としてコピーし、接続先と実際のデプロイ名を記入してください。カーネルの作業フォルダーをこのノートブックのあるフォルダーにして実行します。`.env` の値は既存の環境変数を上書きします。変更後は 0-2 と 0-3 を再実行してください。

認証方式と API キーは共通です。接続先とデプロイ名は API ごとに分けているため、同じ `.env` を編集し直さずに両方を実行できます。

| 環境変数 | 使用先 | 内容 |
| --- | --- | --- |
| `USE_ENTRA_ID` | 共通 | 省略時は `true`。API キーを使う場合は `false` |
| `AZURE_API_KEY` | 共通 | API キー認証時のみ使用。未設定なら 0-3 で非表示入力します |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI Decisions API | Azure OpenAI リソースのルート URL。例: `https://YOUR-RESOURCE-NAME.openai.azure.com` |
| `AZURE_OPENAI_DECISIONS_DEPLOYMENT` | Azure OpenAI Decisions API | デプロイ名。省略時は `gpt-6-luna` |
| `MICROSOFT_DECISION_ENDPOINT` | このノートブック | Foundry リソースのルート URL。デプロイ詳細ページから取得します。未設定なら入力します |
| `MICROSOFT_DECISION_DEPLOYMENT` | このノートブック | デプロイ名。省略時は `Microsoft-Decision-1` |

接続先には `/openai/v1/` や `/providers/microsoft/v1/systemone` を含めません。必要なパスは各ノートブックが追加します。API キーを共用できるのは同じリソースの場合だけです。別リソースのキーが必要なら `.env` の `AZURE_API_KEY` を空欄にし、各ノートブックの 0-3 で対応するキーを入力してください。Microsoft Entra ID 認証では、両リソースへの利用権限が必要です。`.env` は Git の管理対象外です。

出典: [エンドポイントとデプロイ名](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#call-the-decision-api)

In [ ]:
import getpass
import json
import os
from urllib.parse import urlsplit
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_ENDPOINT = (os.getenv('MICROSOFT_DECISION_ENDPOINT') or input('Foundry リソースのルート URL: ')).strip().rstrip('/')
DEPLOYMENT_NAME = os.getenv('MICROSOFT_DECISION_DEPLOYMENT', 'Microsoft-Decision-1').strip()
auth_setting = os.getenv('USE_ENTRA_ID', 'true').strip().lower()
if auth_setting not in ('1', 'true', 'yes', '0', 'false', 'no'):
    raise ValueError('USE_ENTRA_ID には true または false を設定してください。')
USE_ENTRA_ID = auth_setting in ('1', 'true', 'yes')
endpoint_parts = urlsplit(AZURE_ENDPOINT)
if (endpoint_parts.scheme != 'https' or not endpoint_parts.hostname
        or endpoint_parts.path or endpoint_parts.query or endpoint_parts.fragment
        or endpoint_parts.username is not None or endpoint_parts.password is not None
        or 'YOUR-RESOURCE-NAME' in AZURE_ENDPOINT or not DEPLOYMENT_NAME):
    raise ValueError('MICROSOFT_DECISION_ENDPOINT に実際のリソースのルート URL、MICROSOFT_DECISION_DEPLOYMENT にデプロイ名を設定してください。')
DECISION_URL = AZURE_ENDPOINT + '/providers/microsoft/v1/systemone'
print('デプロイ:', DEPLOYMENT_NAME)

### 0-3. 認証

Microsoft Entra ID 認証では、利用権限のあるアカウントで Azure CLI などに事前にサインインしてください。スコープは `https://cognitiveservices.azure.com/.default` です。API キーを使う場合も、キー自体は表示しません。

出典: [認証](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#call-the-decision-api)

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

if USE_ENTRA_ID:
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default'
    )
else:
    API_KEY = os.getenv('AZURE_API_KEY') or getpass.getpass('AZURE_API_KEY: ')
    if not API_KEY:
        raise ValueError('API キーを設定してください。')
print('認証:', 'Microsoft Entra ID' if USE_ENTRA_ID else 'API キー')

### 0-4. 最小限の呼び出し関数

各機能セルから同じ HTTP 処理を呼べるように、`decide(state, questions)` を定義します。送信先は `POST /providers/microsoft/v1/systemone`、本文は `model`・`state`・`questions` です。独自クラスやレスポンスの変換処理は使わず、生の JSON を返します。

このセルでは API を呼びません。自動再試行はせず、HTTP エラー時は本文を表示して停止します。認証トークンは呼び出し時に取得します。

出典: [Decision API の呼び出し](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#call-the-decision-api)

In [ ]:
import httpx

def decide(state, questions):
    headers = {'Authorization': 'Bearer ' + token_provider()} if USE_ENTRA_ID else {'api-key': API_KEY}
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            DECISION_URL,
            headers=headers,
            json={'model': DEPLOYMENT_NAME, 'state': state, 'questions': questions},
        )
    if response.is_error:
        print(response.text)
    response.raise_for_status()
    return response.json()

### 0-5. 結果をグラフで可視化する準備

取得済みの回答を横棒グラフにする関数を定義します。このセルでは API を呼びません。`noul` は返された確率 `p` と補数 `1-p`、`choice` と `score` は選択肢・段階ごとの確率を表示します。スコアと、応答に含まれる場合の `confidence` は見出しに表示します。

複数質問は質問ごとにグラフを分けます。スコアの質問定義を渡すと、段階番号に加えて基準も表示します。拒否は「判定が拒否されました」と表示し、確率 0 に置き換えません。未知の形式はエラーにして、表示済みの JSON で確認できるようにします。日本語フォントが見つからない場合は警告を表示します。

出典: [質問タイプと回答](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#choose-a-question-type)、[レスポンス形式の参考](https://learn.microsoft.com/azure/databricks/machine-learning/model-serving/query-systemone-models)、[Matplotlib の横棒グラフ](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html)

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_candidates = ['Hiragino Sans', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP', 'Noto Sans JP', 'IPAexGothic']
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
japanese_font = next((name for name in font_candidates if name in available_fonts), None)
if japanese_font:
    plt.rcParams['font.family'] = japanese_font
else:
    print('日本語フォントが見つかりません。グラフの日本語が表示されない場合があります。')
plt.rcParams['axes.unicode_minus'] = False

def plot_results(result, questions=None):
    answers = result['answers']
    if not isinstance(answers, dict) or not answers:
        raise ValueError('answers は空でないオブジェクトにしてください。')
    fig, axes = plt.subplots(len(answers), 1, figsize=(9, 3.5 * len(answers)), squeeze=False)
    for ax, (name, answer) in zip(axes[:, 0], answers.items()):
        kind = answer['type']
        title = f'{name} ({kind})'
        if kind == 'refusal':
            print(f'{name}: 判定が拒否されました。')
            ax.text(0.5, 0.5, f'{name}\n判定が拒否されました', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        if kind == 'noul':
            p = answer['noul']
            labels, values = ['はい', 'いいえ'], [p, 1 - p]
        elif kind in ('choice', 'score'):
            probabilities = answer['probabilities']
            if isinstance(probabilities, dict):
                items = list(probabilities.items())
            elif isinstance(probabilities, list):
                items = [(item['value'], item['probability']) for item in probabilities]
            else:
                raise TypeError(f'{name}: probabilities はオブジェクトまたは配列にしてください。')
            labels = [str(value) for value, _ in items]
            if kind == 'score':
                levels = questions[name]['criteria'] if questions is not None else None
                labels = [f'{value}: {levels[int(value)]}' if levels is not None else f'段階 {value}' for value, _ in items]
            values = [probability for _, probability in items]
            selected = answer['choice'] if kind == 'choice' else f"{answer['score']:.3f}"
            title += f'\n{kind}={selected}'
            if answer.get('confidence') is not None:
                title += f", confidence={answer['confidence']:.3f}"
        else:
            raise ValueError(f'未対応の回答形式です: {kind}')
        if not values or not all(math.isfinite(value) and 0 <= value <= 1 for value in values):
            raise ValueError(f'{name}: 確率は 0〜1 の有限値で指定してください。')
        bars = ax.barh(range(len(labels)), values, tick_label=labels, color='#2b6cb0')
        ax.bar_label(bars, fmt='%.3f', padding=3)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.12)
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_xlabel('確率')
        ax.set_title(title)
    fig.tight_layout()
    plt.show()

## Part 1: 3 つの質問タイプとレスポンスの構造

### 1-1. noul: yes である確率

`noul` は条件が成り立つ確率（0〜1）を返します。`questions` は質問名をキーに持つオブジェクトで、回答も `answers` の同じ質問名で参照します。返金を求めているかの推定であり、返金してよいかの判定ではありません。

生の回答を表示するため、実際の `type` と数値フィールドを確認してください。拒否や解釈できない回答を、否定やスコア 0 に置き換えないでください。Foundry の公式ページでは具体的なレスポンス例が限定されるため、この版では独自パーサーを使いません。

出典: [Noul](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#use-noul-for-gates-and-filters)

In [ ]:
Q_REFUND = {'type': 'noul', 'instructions': 'お客様は返金を求めていますか？'}
noul_result = decide(
    'カードが同じ注文で二重に請求されました。余分な請求分を返金してください。',
    {'refund_requested': Q_REFUND},
)
print(json.dumps(noul_result['answers'], ensure_ascii=False, indent=2))
plot_results(noul_result)

### 1-2. noul の criteria: true と false の基準

`criteria` は任意です。返金を明示的に求めている場合と、状況確認だけの場合の基準を追加できます。同じ文章に対し、基準なし・基準ありの 2 問を 1 リクエストで比較します。特定方向に確率が変わることや精度向上を保証する例ではありません。

出典: [Noul の判定基準](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#use-noul-for-gates-and-filters)

In [ ]:
criteria_result = decide(
    '二重に請求されたようです。まず状況を確認してください。返金はまだ求めていません。',
    {
        'plain': Q_REFUND,
        'with_criteria': {
            **Q_REFUND,
            'criteria': {
                'true': '返金または払い戻しを、はっきりと求めている。',
                'false': '状況の確認や問い合わせだけで、返金を求めていない。',
            },
        },
    },
)
print(json.dumps(criteria_result['answers'], ensure_ascii=False, indent=2))
plot_results(criteria_result)

### 1-3. choice: 固定の選択肢から 1 つ選ぶ

`criteria` に選択肢名と説明のオブジェクトを指定します。回答には選択した値と各選択肢の確率が含まれます。この例では請求・技術・サポートの担当チームを選びます。[Azure OpenAI 版](./azure_openai_decisions_api.ipynb) の `choices` 配列とは形式が異なります。

出典: [Choice](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#use-choice-for-classification-and-routing)

In [ ]:
Q_TEAM = {
    'type': 'choice',
    'instructions': 'この問い合わせは、どのチームが対応すべきですか？',
    'criteria': {
        'billing': '請求、支払い、返金に関する内容。',
        'engineering': '不具合、エラー、障害に関する内容。',
        'support': '使い方やアカウントに関する質問。',
    },
}
choice_state = 'API を呼び出すたびに 500 エラーが返ります。'
choice_result = decide(choice_state, {'team': Q_TEAM})
print(json.dumps(choice_result['answers'], ensure_ascii=False, indent=2))
plot_results(choice_result)

In [ ]:
choice_state = '2段階認証を有効にしたいです。'
choice_result = decide(choice_state, {'team': Q_TEAM})
print(json.dumps(choice_result['answers'], ensure_ascii=False, indent=2))
plot_results(choice_result)

### 1-4. score: 順序のある目盛りで採点する

`criteria` に段階を低い順の配列で渡します。`choice` のように 1 カテゴリを選ぶのではなく、順序のある尺度に対するスコアと段階ごとの確率を確認します。隣り合う段階の違いが分かる基準を記述します。

出典: [Score](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#use-score-for-ranking-and-grading)

In [ ]:
Q_SEVERITY = {
    'type': 'score',
    'instructions': 'この障害の深刻度はどの程度ですか？',
    'criteria': [
        '軽微: 機能に影響がない。',
        '軽度: 回避策がある。',
        '重大: 一部のお客様の業務が止まっている。',
        '致命的: すべてのお客様の業務が止まっている。',
    ],
}
score_result = decide(
    'EU のすべてのお客様で、午前 9 時からチェックアウトが停止しています。',
    {'severity': Q_SEVERITY},
)
print(json.dumps(score_result['answers'], ensure_ascii=False, indent=2))
plot_results(score_result, {'severity': Q_SEVERITY})

### 1-5. score の計算を手元で確かめる

API は呼びません。同じ System One 形式を扱う Databricks の公式文書を参考に、段階の添字の確率加重平均を計算します。Foundry とは別のホスティングであり、レスポンスの細部まで同一とは断定しません。以下は計算用の例示データで、実 API の結果ではありません。

1 に集中した分布と 0・2 に半分ずつ割れた分布は、どちらもスコアが 1 です。棒グラフを並べて分布を比較し、破線でスコアの位置を表示します。スコアだけでなく確率分布も確認する理由になります。

出典: [Databricks の System One API](https://learn.microsoft.com/azure/databricks/machine-learning/model-serving/query-systemone-models)

In [ ]:
cases = {
    '計算例': [0.1, 0.7, 0.2],
    '1 段階目に集中': [0.0, 1.0, 0.0],
    '0 と 2 に分散': [0.5, 0.0, 0.5],
}
fig, axes = plt.subplots(1, len(cases), figsize=(12, 3.5), sharey=True)
for ax, (label, probabilities) in zip(axes, cases.items()):
    score = sum(i * p for i, p in enumerate(probabilities))
    print(f'{label}: probabilities={probabilities}, score={score:.2f}')
    ax.bar(range(len(probabilities)), probabilities, color='#2b6cb0')
    ax.axvline(score, color='#c05621', linestyle='--', label='スコア')
    ax.set_title(f'{label}\nscore={score:.2f}')
    ax.set_xticks(range(len(probabilities)))
    ax.set_xlabel('段階の添字')
    ax.set_ylim(0, 1.05)
    ax.legend()
axes[0].set_ylabel('確率')
fig.tight_layout()
plt.show()

### 1-6. リクエストとレスポンスの生の JSON

1-3 で送った本文を再構成し、保存済みのレスポンスと並べて表示します。追加の API 呼び出しはありません。`answers` は質問名をキーに持つオブジェクト、`usage` はトークン使用量です。応答の `model` は基盤モデルを表し、指定したデプロイ名と異なる場合があります。

出典: [リクエストとモデル名](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#call-the-decision-api)、[回答](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#choose-a-question-type)

In [ ]:
payload = {'model': DEPLOYMENT_NAME, 'state': choice_state, 'questions': {'team': Q_TEAM}}
print('--- リクエスト本文 ---')
print(json.dumps(payload, ensure_ascii=False, indent=2))
print('--- レスポンス本文 ---')
print(json.dumps(choice_result, ensure_ascii=False, indent=2))

## Part 2: 入力と質問のバリエーション

### 2-1. state に JSON を渡す

`state` には文章だけでなく JSON オブジェクトも渡せます。問い合わせ本文だけの場合と、同じ件での連絡回数を含めた場合を同じ質問で比較するため、このセルは 2 回呼び出します。情報追加による確率の変化や精度向上を保証するものではありません。

出典: [テキストと JSON の入力](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision)

In [ ]:
ticket = {
    'message': 'チェックアウトがまた止まっています。',
    'customer_contact_count': 3,
    'plan': 'enterprise',
}
Q_REPEAT = {'type': 'noul', 'instructions': 'このお客様は、以前にも同じ件で問い合わせていますか？'}
repeat_questions = {'repeat_contact': Q_REPEAT}
text_result = decide(ticket['message'], repeat_questions)
json_result = decide(ticket, repeat_questions)
print('文章だけ:', json.dumps(text_result['answers'], ensure_ascii=False, indent=2))
plot_results(text_result)
print('JSON 入力:', json.dumps(json_result['answers'], ensure_ascii=False, indent=2))
plot_results(json_result)

### 2-2. state に会話の配列を渡す

顧客と担当者の発言を JSON 配列で渡し、担当者が返金を約束したかを質問します。`role` と `content` はこの例の入力データであり、Chat Completions のメッセージ仕様を指定するものではありません。

出典: [Foundry の入力](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision)、[Databricks の配列入力](https://learn.microsoft.com/azure/databricks/machine-learning/model-serving/query-systemone-models)

In [ ]:
conversation = [
    {'role': 'customer', 'content': '二重に請求されています。'},
    {'role': 'agent', 'content': '余分な請求分は、本日中に返金いたします。'},
]
conversation_result = decide(conversation, {
    'agent_promises_refund': {'type': 'noul', 'instructions': '担当者は、お客様に返金を約束していますか？'},
})
print(json.dumps(conversation_result['answers'], ensure_ascii=False, indent=2))
plot_results(conversation_result)

### 2-3. 1 回のリクエストで複数の質問をする

ここでは Part 1・2-1 の質問を再利用し、同じ JSON に対して担当チーム・深刻度・再問い合わせの 3 点を 1 回で評価します。`questions` と `answers` の質問名を見比べてください。別の質問の回答を後続の質問に渡す処理ではありません。

出典: [質問オブジェクトと回答](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#choose-a-question-type)

In [ ]:
questions = {'team': Q_TEAM, 'severity': Q_SEVERITY, 'repeat_contact': Q_REPEAT}
multi_result = decide(
    {'message': 'チェックアウトがまた止まっています。これで 3 回目の連絡です。', 'customer_contact_count': 3},
    questions,
)
print('質問名:', list(questions))
print('回答名:', list(multi_result['answers']))
print(json.dumps(multi_result, ensure_ascii=False, indent=2))
plot_results(multi_result, questions)

### 2-4. 説明のない選択肢（null）

選択肢の名前だけで意味が明確な場合に、説明を `None`（送信時の JSON では `null`）にする例です。

**要確認:** この形式は Databricks の System One 文書に記載されていますが、Foundry の Decision-1 文書には明記されていません。利用環境で受け付けられるかを確認するセルです。未対応の場合はエラー本文を表示して停止し、成功した結果に置き換えることはしません。

出典: [Databricks の choice の説明](https://learn.microsoft.com/azure/databricks/machine-learning/model-serving/query-systemone-models)、[Foundry の Choice](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision#use-choice-for-classification-and-routing)

In [ ]:
null_result = decide('日経平均株価が前日より 300 円上がりました。', {
    'category': {
        'type': 'choice',
        'instructions': 'この文章の話題は、どれに最も近いですか？',
        'criteria': {'天気': None, 'スポーツ': None, '経済': None},
    },
})
print(json.dumps(null_result['answers'], ensure_ascii=False, indent=2))
plot_results(null_result)